In [1]:
import torch 
import sys
from datasets.graph_datasets.graph_heat_dataset import HeatGraphDataset
import yaml
from models.forecasting.GNO import GNO
from torch_geometric.loader import DataLoader
from datasets.graph_datasets.graph_data_utils import partition_domain_into_subgraphs, get_subgraph_grid_size
sys.path.append('...')

/Users/louisgodtfredsen/Desktop/Coding Projects/ML-for-PDEs/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load configs
with open("../checkpoints/gno_heat/20260912_2148/model_configs.yaml", "r") as file:
    cfg = yaml.safe_load(file)

# Load in sample to run inference on

In [3]:
data_path ='../data/test_data/heat_equation_m64_h0_minmax_N200.pt'
input_data = torch.load(data_path)
simulation_idx = 0
simulation_frame = 0
field_keys = list(input_data.keys())[1:]

X, X_t1 = input_data['X'][simulation_idx,simulation_frame], input_data['X'][simulation_idx,simulation_frame+1]
pde_params = [float(input_data[k][simulation_idx]) for k in field_keys]

H, W = X.shape[-1], X.shape[-2] 
x_indices = torch.tensor([x for x in range(W)])
y_indices = torch.tensor([x for x in range(H)])
node_grid_indices = torch.cartesian_prod(x_indices, y_indices) # Collection of (x, y) grid indices
node_spatial_pos = torch.cartesian_prod(x_indices / W, y_indices / H) # Collection of (x, y) spatial positions

dataset = HeatGraphDataset('../data/test_data/heat_equation_m64_h0_minmax_N200.pt',
                           list(input_data.keys())[1:],
                           r = cfg['radius'],
                           bc ='periodic',
                           sub_graph_size = cfg['sub_graph_size'])

In [4]:
get_subgraph_grid_size(200)

(10, 20)

In [5]:
subgraphs = partition_domain_into_subgraphs(X, X_t1, H, W, num_subgraph_nodes=200, r = 0.025, boundary_condition='periodic', pde_params=pde_params)

Current start h location 0
Current end  h location 10
Current start w location 0
Current end  w location 20
Current start w location 20
Current end  w location 40
Current start w location 40
Current end  w location 60
Current start w location -20
Current end w location 63
Current start h location 10
Current end  h location 20
Current start w location 0
Current end  w location 20
Current start w location 20
Current end  w location 40
Current start w location 40
Current end  w location 60
Current start w location -20
Current end w location 63
Current start h location 20
Current end  h location 30
Current start w location 0
Current end  w location 20
Current start w location 20
Current end  w location 40
Current start w location 40
Current end  w location 60
Current start w location -20
Current end w location 63
Current start h location 30
Current end  h location 40
Current start w location 0
Current end  w location 20
Current start w location 20
Current end  w location 40
Current start w

In [6]:
subgraphs

[Data(x=[200, 3], edge_index=[2, 1424], edge_attr=[1424, 5], y=[200, 1], num_nodes=200),
 Data(x=[200, 3], edge_index=[2, 1424], edge_attr=[1424, 5], y=[200, 1], num_nodes=200),
 Data(x=[200, 3], edge_index=[2, 1424], edge_attr=[1424, 5], y=[200, 1], num_nodes=200),
 Data(x=[200, 3], edge_index=[2, 1424], edge_attr=[1424, 5], y=[200, 1], num_nodes=200),
 Data(x=[200, 3], edge_index=[2, 1424], edge_attr=[1424, 5], y=[200, 1], num_nodes=200),
 Data(x=[200, 3], edge_index=[2, 1424], edge_attr=[1424, 5], y=[200, 1], num_nodes=200),
 Data(x=[200, 3], edge_index=[2, 1424], edge_attr=[1424, 5], y=[200, 1], num_nodes=200),
 Data(x=[200, 3], edge_index=[2, 1424], edge_attr=[1424, 5], y=[200, 1], num_nodes=200),
 Data(x=[200, 3], edge_index=[2, 1424], edge_attr=[1424, 5], y=[200, 1], num_nodes=200),
 Data(x=[200, 3], edge_index=[2, 1424], edge_attr=[1424, 5], y=[200, 1], num_nodes=200),
 Data(x=[200, 3], edge_index=[2, 1424], edge_attr=[1424, 5], y=[200, 1], num_nodes=200),
 Data(x=[200, 3], edg

In [7]:
X[10, :20]

tensor([1.6797e-01, 1.2390e-01, 8.7942e-02, 6.0083e-02, 3.9540e-02, 2.5093e-02,
        1.5388e-02, 9.1480e-03, 5.3019e-03, 3.0225e-03, 1.7181e-03, 9.9235e-04,
        5.9498e-04, 3.7686e-04, 2.5361e-04, 1.7984e-04, 1.3215e-04, 9.8909e-05,
        7.4403e-05, 5.5765e-05])

In [8]:
subgraphs[0].x

tensor([[0.0000e+00, 0.0000e+00, 9.1053e-02],
        [0.0000e+00, 1.5625e-02, 6.7491e-02],
        [0.0000e+00, 3.1250e-02, 4.8079e-02],
        [0.0000e+00, 4.6875e-02, 3.2924e-02],
        [0.0000e+00, 6.2500e-02, 2.1678e-02],
        [0.0000e+00, 7.8125e-02, 1.3730e-02],
        [0.0000e+00, 9.3750e-02, 8.3721e-03],
        [0.0000e+00, 1.0938e-01, 4.9211e-03],
        [0.0000e+00, 1.2500e-01, 2.7948e-03],
        [0.0000e+00, 1.4062e-01, 1.5397e-03],
        [0.0000e+00, 1.5625e-01, 8.2845e-04],
        [0.0000e+00, 1.7188e-01, 4.4030e-04],
        [0.0000e+00, 1.8750e-01, 2.3523e-04],
        [0.0000e+00, 2.0312e-01, 1.2936e-04],
        [0.0000e+00, 2.1875e-01, 7.5092e-05],
        [0.0000e+00, 2.3438e-01, 4.6783e-05],
        [0.0000e+00, 2.5000e-01, 3.1251e-05],
        [0.0000e+00, 2.6562e-01, 2.2019e-05],
        [0.0000e+00, 2.8125e-01, 1.6013e-05],
        [0.0000e+00, 2.9688e-01, 1.1800e-05],
        [1.5625e-02, 0.0000e+00, 1.0901e-01],
        [1.5625e-02, 1.5625e-02, 8

In [15]:
subgraphs[4].x[:20, -1]

tensor([1.6797e-01, 1.2390e-01, 8.7942e-02, 6.0083e-02, 3.9540e-02, 2.5093e-02,
        1.5388e-02, 9.1480e-03, 5.3019e-03, 3.0225e-03, 1.7181e-03, 9.9235e-04,
        5.9498e-04, 3.7686e-04, 2.5361e-04, 1.7984e-04, 1.3215e-04, 9.8909e-05,
        7.4403e-05, 5.5765e-05])

# Load Model

In [10]:
gno_model = GNO(optimiser = cfg['optimiser'], 
                 learning_rate = cfg['learning_rate'], 
                 num_node_input_features = cfg['num_node_input_features'],
                 num_edge_features = cfg['num_edge_features'], 
                 num_latent_dim = cfg['num_latent_dim'], 
                 output_dim = cfg['output_dim'],
                 num_gno_layers = cfg['num_gno_layers'],
                 kernel_ffn_layers = cfg['kernel_ffn_layers'],
                 kernel_ffn_dropout = cfg['kernel_ffn_dropout'], 
                 GNO_layer_activation = cfg['gno_layer_activation'])

model_path = '../checkpoints/gno_heat/20260912_2148/gno-epoch=0009-val_loss=0.0000.ckpt'
gno_model.load_state_dict(torch.load(model_path)['state_dict'])

<All keys matched successfully>

In [11]:
import torch

A = torch.randint(low = 0, high=5, size = (5,5))

In [12]:
A

tensor([[1, 3, 2, 2, 3],
        [4, 1, 4, 0, 2],
        [3, 1, 0, 0, 2],
        [0, 0, 2, 0, 3],
        [4, 3, 0, 0, 2]])

In [13]:
A[0:3, 1:4]

tensor([[3, 2, 2],
        [1, 4, 0],
        [1, 0, 0]])

In [14]:
A.flatten()

tensor([1, 3, 2, 2, 3, 4, 1, 4, 0, 2, 3, 1, 0, 0, 2, 0, 0, 2, 0, 3, 4, 3, 0, 0,
        2])